In [78]:

path = "D:/02_사업관리/2025년/데이터 분석 컨설팅/20251022_체육정책과/02_자료수집/"
path2 = "D:/02_사업관리/2025년\데이터 분석 컨설팅/20251022_체육정책과/03_집계결과/"

import pandas as pd
import numpy as np

df1 = pd.read_csv(path + "kt_d_living_emd_inflow_202410.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str}, encoding="utf-8")
df2 = pd.read_csv(path + "kt_d_living_emd_inflow_202510.csv", sep="|", dtype={'std_ymd': str, 'hcode': str, 'inflow_cd': str}, encoding="utf-8")
df3 = pd.read_csv(path + "bccd_d_emd_inflow_202410.csv", sep="|", dtype={'hcode': str, 'ages': str}, encoding="utf-8")
df4 = pd.read_csv(path + "bccd_d_emd_inflow_202510_TMP.txt", sep="|", dtype={'std_ymd': str, 'hcode': str, 'ages': str}, encoding="utf-8")

code = pd.read_csv("D:/02_사업관리/2025년/데이터 분석 컨설팅/코드자료/행정동코드_대상지역.csv", dtype={'gcode': str, 'hcode': str}, encoding="utf-8").dropna()
age_cd = pd.read_csv("D:/02_사업관리/2025년/데이터 구매 및 활용/12_적재확인/코드자료/bccd_age_code.csv", sep="|", dtype={'age_cd': str}, encoding="utf-8")
age_cd.columns = ['ages', 'agenm']


#### 카드이용 변화분석

In [ ]:
# 2024년 데이터에서 동일지역, 동일기간 추출
cond1 = (df3['std_ymd'] >= 20241017) & (df3['std_ymd'] <= 20241023) & (df3['hcode'].str.startswith('26260'))
cond2 = (df3['std_ymd'] >= 20241017) & (df3['std_ymd'] <= 20241023) & (df3['hcode'].str.startswith('26470'))
df31 = df3[cond1].copy()
df32 = df3[cond2].copy()
df3 = pd.concat([df31, df32], ignore_index=True)
# print(df1.shape) # (228454, 10)
# print(df2.shape) # (246357, 10)

# 행정동코드 변환
df3 = df3.merge(code, how="left", on="hcode")
df4 = df4.merge(code, how="left", on="hcode")

# 2024년, 2025년 성별 이용금액, 이용건수
df3_gen = df3.groupby(['std_ymd','gname','hname','gender'])[['amt', 'cnt']].sum().reset_index().sort_values(['std_ymd'], ascending=True)
df3_gen.columns = ['std_ymd', 'gname', 'hname', 'gender', 'amt24', 'cnt24']
df4_gen = df4.groupby(['std_ymd','gname', 'hname','gender'])[['amt', 'cnt']].sum().reset_index().sort_values(['std_ymd'], ascending=True)
df4_gen.columns = ['std_ymd', 'gname', 'hname', 'gender', 'amt25', 'cnt25']

df3_gen.to_csv(path + "2024년 기간별_성별_카드이용 현황.csv", sep=",", encoding="utf-8", index=False)
df4_gen.to_csv(path + "2025년 기간별_성별_카드이용 현황.csv", sep=",", encoding="utf-8", index=False)

# 2024년, 2025년 연령별 이용금액, 이용건수
df3_age = df3.groupby(['std_ymd','gname','hname','ages'])[['amt', 'cnt']].sum().reset_index().sort_values(['std_ymd'], ascending=True)
df3_age.columns = ['std_ymd', 'gname', 'hname', 'ages', 'amt24', 'cnt24']
df3_age = df3_age.merge(age_cd, how="left", on='ages')
df3_age = df3_age[['std_ymd', 'gname', 'hname', 'agenm', 'amt24', 'cnt24']]
df4_age = df4.groupby(['std_ymd', 'gname', 'hname', 'ages'])[['amt', 'cnt']].sum().reset_index().sort_values(['std_ymd'], ascending=True)
df4_age.columns = ['std_ymd','gname','hname','ages','amt25','cnt25']
df4_age = df4_age.merge(age_cd, how="left", on='ages')
df4_age = df4_age[['std_ymd', 'gname', 'hname', 'agenm', 'amt25', 'cnt25']]

df3_age.to_csv(path2 + "2024년 기간별_연령_카드이용 현황.csv", sep=",", encoding="utf-8", index=False)
df4_age.to_csv(path2 + "2025년 기간별_연령_카드이용 현황.csv", sep=",", encoding="utf-8", index=False)

#### 유입인구 변화분석

In [90]:

def main(df):
    df['std_ymd'] = df['std_ymd'].astype('datetime64[ns]')
    cond1 = (df['std_ymd'].dt.day >= 17) & (df['std_ymd'].dt.day <= 23)
    cond2 = (df['hcode'].str.startswith('26260')) & (df['hcode'].str.startswith('26470'))
    cond3 = (~df['inflow_cd'].str.startswith('26'))
    df = df[cond1 & cond2 | cond3].copy()

    df = df.merge(code, "left", "hcode")

    df.loc[df['gen_age'].str.startswith('m'), 'gender'] = '남성'
    df.loc[df['gen_age'].str.startswith('m'), 'gender'] = '여성'

    df.loc[df['gen_age'].str.contains('0009|1014|1519'), 'ages'] = '20대미만'
    df.loc[df['gen_age'].str.contains('2024|2529'), 'ages'] = '20대'
    df.loc[df['gen_age'].str.contains('3034|3539'), 'ages'] = '30대'
    df.loc[df['gen_age'].str.contains('4044|4549'), 'ages'] = '40대'
    df.loc[df['gen_age'].str.contains('5054|5559'), 'ages'] = '50대'
    df.loc[df['gen_age'].str.contains('6064|6569'), 'ages'] = '60대'
    df.loc[df['gen_age'].str.contains('7000'), 'ages'] = '70대이상'

    df['all_pop'] = df['h_pop'] + df['w_pop'] + df['v_pop']

    gender = df.groupby(['std_ymd','gname','hname','gender'])['all_pop'].sum().reset_index().sort_values('std_ymd', ascending=True)
    ages = df.groupby(['std_ymd','gname','hname','ages'])['all_pop'].sum().reset_index().sort_values('std_ymd', ascending=True)

    gender.to_csv(path2 + f"{df['std_ymd'].astype(str)[0][:4]}년 성별 역외유입인구.csv", sep=",", encoding="utf-8", index=False)
    ages.to_csv(path2 + f"{df['std_ymd'].astype(str)[0][:4]}년 연령 역외유입인구.csv", sep=",", encoding="utf-8", index=False)

main(df1)
main(df2)
